# Milestone 4 · Notebook 1b: The Test Sets

**Project:** Deepfake Voice Detection for Indian Languages · Group-3

This notebook adds the test sets to the data built by `01_data_preparation.ipynb` (same sources, pinned revisions,
speaker split and preprocessing). The test sets are fixed now and reused unchanged in Milestone 5.

| Test split | Contents | Full set | Milestone-4 sample (`m4_eval`) |
| --- | --- | ---: | ---: |
| `test_indomain` | Test speakers (15% split): Kathbath 1,000 + IndicSynth 1,000 per language | 6,000 | 500 + 500 |
| `test_unseen_gen` | All of our generated set (Indic Parler-TTS, IndicF5, MMS-TTS) + the matching Kathbath test-speaker recordings | ~8,640 | 500 + 500 |
| `test_phone` | `test_indomain` through a simulated phone channel (300–3,400 Hz, 8 kHz, G.711) | 6,000 | 500 + 500 |
| `test_itw` | In-the-Wild, real-world English | 31,779 | 500 + 500 |
| `test_asvspoof` | ASVspoof 2019 LA eval, attacks A07–A19 (none seen in training) | 71,237 | 500 + 500 |

**Two stages.** By default the notebook prepares only the Milestone-4 sample, a fixed, seeded subset of each full set,
which Milestone 4 scores. With the environment variable `M4_FULL=1` it prepares every clip of the full sets for
Milestone 5. The sample is chosen from the full listing in both stages, so it is the same clips either way, and the
full run only adds the rest.

## 1. Setup

In [1]:
import warnings

warnings.filterwarnings("ignore", message="IProgress not found")

import os
import sys
import time
import json
import shutil
import tarfile
import zipfile
import zlib
from collections import Counter, deque
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
from huggingface_hub import HfFileSystem, hf_hub_download

REPO = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(REPO))
from src.augment import phone_channel
from src.preprocess import TARGET_SR, normalise, sha1, to_pcm16

pd.set_option("display.width", 200)
fs = HfFileSystem()

In [2]:
SEED = 42
LANGUAGES = {"hi": "Hindi", "ta": "Tamil", "te": "Telugu"}

INDICSYNTH = "datasets/vdivyasharma/IndicSynth@c0a10386b723717aff682f757bd67f72983f269f"
KATHBATH = "datasets/ai4bharat/Kathbath@5b9e92849222026d9141acba4e8434fe816396bf"
GENERATED = "datasets/TarikaHedaoo/dsai-group3-fake-speech@b9db125b305701982c76b20ef613217859f43c42"
ASVSPOOF = "datasets/SpeechAntiSpoofingBenchmarks/ASVspoof2019_LA@401e90d3efddb3a2b2ee5f3ee53b737b603d7eae"
IN_THE_WILD = "datasets/mueller91/In-The-Wild@eee168f92c367f8c82ff2cf42b6f61e362fd6211"
KATHBATH_TEST_TAR = "https://objectstore.e2enetworks.net/indic-superb/kathbath/clean/testunk_audio.tar"

INDOMAIN_PER_LANGUAGE = 1000     # real per language; fakes the same, half per generator
M4_EVAL_PER_LABEL = 500          # the Milestone-4 sample of each test split
CLIPS_PER_ROW_GROUP = 25
RAW_PER_SOURCE = 10
FULL = bool(os.environ.get("M4_FULL"))  # False: only the Milestone-4 sample; True: every clip (Milestone 5)

DATA = REPO / "data" / "m4"
RAW, PROCESSED, MANIFESTS = DATA / "raw", DATA / "processed", DATA / "manifests"
CACHE = REPO / "data" / "cache_m4"   # source listings, reused by the full run; not uploaded
CACHE.mkdir(parents=True, exist_ok=True)
FFMPEG = shutil.which("ffmpeg") or __import__("imageio_ffmpeg").get_ffmpeg_exe()


def rng_for(name):
    """One fixed generator per source, so each choice is the same whatever runs before it."""
    return np.random.default_rng([SEED, zlib.crc32(name.encode())])


speakers = pd.read_csv(MANIFESTS / "speaker_split.csv")
split_of = {(s.language, s.speaker_id): s.split for s in speakers.itertuples()}
print("Stage:", "full test sets (Milestone 5)" if FULL else "Milestone-4 sample", "| data folder:", DATA)

Stage: Milestone-4 sample | data folder: /home/himanshu/myproject/dsai-project/data/m4


## 2. Helpers

The same helpers as notebook 1, plus `subsample`, which takes the Milestone-4 sample from a full listing.

In [3]:
rows, dropped, raw_kept = [], [], Counter()


def save(clip, data, raw_name):
    """Preprocess one clip, write it as a 16-bit WAV and add a manifest row.

    clip: dict with clip_id, dataset, language, label, generator, speaker_id, ref_speaker_id, gender, text, split.
    """
    pcm, sr = to_pcm16(data)
    if pcm is None:
        dropped.append({**clip, "reason": "shorter than 1 s after trimming"})
        return
    out = PROCESSED / clip["dataset"] / clip["language"] / f"{clip['clip_id']}.wav"
    out.parent.mkdir(parents=True, exist_ok=True)
    sf.write(out, pcm, TARGET_SR, subtype="PCM_16")
    raw_path = ""
    if raw_kept[(clip["dataset"], clip["language"])] < RAW_PER_SOURCE:  # keep a few originals
        raw_kept[(clip["dataset"], clip["language"])] += 1
        raw = RAW / clip["dataset"] / clip["language"] / raw_name
        raw.parent.mkdir(parents=True, exist_ok=True)
        raw.write_bytes(data)
        raw_path = str(raw.relative_to(DATA))
    rows.append({**clip, "path": str(out.relative_to(DATA)), "raw_path": raw_path,
                 "duration_s": round(len(pcm) / TARGET_SR, 3), "orig_sr": sr, "sha1": sha1(pcm)})


def restore(clip, raw_name, orig_sr):
    """If this clip was already processed by an earlier, interrupted run, add its manifest row from the stored file.

    Returns True when the clip is already done, so it isn't downloaded again.
    """
    out = PROCESSED / clip["dataset"] / clip["language"] / f"{clip['clip_id']}.wav"
    if not out.exists():
        return False
    pcm, _ = sf.read(out, dtype="int16")
    raw = RAW / clip["dataset"] / clip["language"] / raw_name
    if raw.exists():
        raw_kept[(clip["dataset"], clip["language"])] += 1
    rows.append({**clip, "path": str(out.relative_to(DATA)), "raw_path": str(raw.relative_to(DATA)) if raw.exists() else "",
                 "duration_s": round(len(pcm) / TARGET_SR, 3), "orig_sr": orig_sr, "sha1": sha1(pcm)})
    return True


def pick(candidates, n):
    """n random rows, at most CLIPS_PER_ROW_GROUP from each randomly chosen (file, row_group)."""
    groups = list(candidates.groupby(["file", "row_group"]).groups.values())
    chosen, total = [], 0
    for g in rng.permutation(len(groups)):
        take = candidates.loc[groups[g]].sample(min(CLIPS_PER_ROW_GROUP, n - total, len(groups[g])),
                                                random_state=int(rng.integers(2**31)))
        chosen.append(take)
        total += len(take)
        if total == n:
            break
    if total < n:
        print(f"  only {total} of {n} available")
    return pd.concat(chosen)


def read_columns(path, columns):
    """The given (small) columns of one parquet file, with each row's row group and position in it."""
    with fs.open(path, cache_type="none") as f:
        parquet = pq.ParquetFile(f, pre_buffer=True)
        part = parquet.read(columns=columns).to_pandas()
        sizes = [parquet.metadata.row_group(i).num_rows for i in range(parquet.metadata.num_row_groups)]
    part["file"] = path
    part["row_group"] = np.repeat(np.arange(len(sizes)), sizes)
    part["row_in_group"] = np.concatenate([np.arange(n) for n in sizes])
    return part


def read_audio(picks, column, parallel=3):
    """Yield (row, audio bytes), downloading each needed row group once.

    At most `parallel` row groups are downloaded or held in memory at a time (row groups can be large).
    """
    def load(key):
        with fs.open(key[0], cache_type="none") as f:
            table = pq.ParquetFile(f, pre_buffer=True).read_row_group(key[1], columns=[column])
        return key, table.column(column).to_pylist()

    keys = list(picks.groupby(["file", "row_group"]).groups)
    with ThreadPoolExecutor(parallel) as pool:
        pending = deque(pool.submit(load, k) for k in keys[:parallel])
        next_key = parallel
        while pending:
            (file, group), audio = pending.popleft().result()
            if next_key < len(keys):
                pending.append(pool.submit(load, keys[next_key]))
                next_key += 1
            for row in picks[(picks.file == file) & (picks.row_group == group)].itertuples():
                yield row, audio[row.row_in_group]["bytes"]
            del audio

In [4]:
def subsample(frame, n, by, seed):
    """n rows with the same share of each `by` group as `frame` (the Milestone-4 sample)."""
    shares = frame.groupby(by).size() / len(frame)
    counts = (shares * n).round().astype(int)
    counts.iloc[0] += n - counts.sum()
    groups = frame.groupby(by)
    return pd.concat([groups.get_group(key).sample(int(count), random_state=seed)
                      for key, count in counts.items() if count > 0])


ORIGINAL_SR = {"indicsynth": 24000, "kathbath": 16000, "in_the_wild": 16000, "asvspoof2019_la": 16000}
m4_eval_ids = set()


def prepare(chosen, sample, clip_of, raw_name_of, orig_sr_of, fetch):
    """Prepare the clips of this stage: all of `chosen` (full run) or only `sample` (Milestone 4).

    clip_of(row) -> manifest fields; fetch(rows) yields (row, audio bytes) for the clips that aren't done yet.
    """
    m4_eval_ids.update(clip_of(r)["clip_id"] for r in sample.itertuples())
    rows_now = chosen if FULL else sample
    todo = rows_now[[not restore(clip_of(r), raw_name_of(r), orig_sr_of(r)) for r in rows_now.itertuples()]]
    for r, data in fetch(todo):
        save(clip_of(r), data, raw_name_of(r))
    return len(rows_now), len(todo)

## 3. `test_indomain`: New Speakers, Known Generators

The 15% of Kathbath training speakers held out as test speakers, and the IndicSynth fakes of those speakers (for
FreeVC24, both speakers must be test speakers).

### 3.1 IndicSynth (fake)

In [5]:
t0 = time.time()
meta_path = REPO / "data" / "raw" / "indicsynth" / "indicsynth_metadata_raw.parquet"
if not meta_path.exists():
    meta_path = hf_hub_download("vdivyasharma/IndicSynth", "metadata.parquet", repo_type="dataset",
                                revision=INDICSYNTH.split("@")[1])
meta = pd.read_parquet(meta_path)
meta["split"] = [split_of[(l, s)] for l, s in zip(meta.language, meta.target_speaker)]
source_split = [split_of[(l, s)] if pd.notna(s) else None for l, s in zip(meta.language, meta.source_speaker)]
usable = meta[[src is None or src == tgt for src, tgt in zip(source_split, meta.split)]].copy()
usable = usable[usable.split == "test_indomain"]
usable["file"] = INDICSYNTH + "/" + usable.language + "/" + usable.file
usable["lang"] = usable.language.map({v: k for k, v in LANGUAGES.items()})

rng = rng_for("indomain-fake")
fakes = pd.concat([pick(usable[(usable.lang == code) & (usable.generator == generator)], INDOMAIN_PER_LANGUAGE // 2)
                   for code in LANGUAGES for generator in ["xtts_v2", "freevc24"]])
fake_sample = subsample(fakes, M4_EVAL_PER_LABEL, ["lang", "generator"], SEED)


def fake_clip(r):
    return {"clip_id": f"indicsynth_{r.lang}_{Path(r.audio_path).stem}", "dataset": "indicsynth", "language": r.lang,
            "label": "fake", "generator": r.generator, "speaker_id": r.target_speaker,
            "ref_speaker_id": r.source_speaker if pd.notna(r.source_speaker) else r.target_speaker,
            "gender": r.gender[0].lower(), "text": r.transcript if isinstance(r.transcript, str) else "",
            "split": "test_indomain"}


n, downloaded = prepare(fakes, fake_sample, fake_clip, lambda r: r.audio_path, lambda r: 24000,
                        lambda todo: read_audio(todo, "audio"))
print(f"IndicSynth test speakers: {len(fakes):,} in the full set, {n:,} prepared ({downloaded:,} downloaded now), "
      f"{(time.time() - t0) / 60:.0f} min")

IndicSynth test speakers: 3,000 in the full set, 500 prepared (0 downloaded now), 0 min


### 3.2 Kathbath (real)

The listing of every Kathbath training file is read once and cached for the full run.

In [6]:
t0 = time.time()
listing = CACHE / "kathbath_listing.parquet"
if listing.exists():
    kathbath = pd.read_parquet(listing)
else:
    paths = [p for folder in ["hindi", "tamil", "telugu"] for p in sorted(fs.glob(f"{KATHBATH}/{folder}/train-*.parquet"))]
    with ThreadPoolExecutor(4) as pool:
        kathbath = pd.concat(pool.map(lambda path: read_columns(path, ["fname", "text", "lang", "gender", "speaker_id"]),
                                      paths), ignore_index=True)
    kathbath.to_parquet(listing)
kathbath["split"] = [split_of.get((LANGUAGES[l], s)) for l, s in zip(kathbath.lang, kathbath.speaker_id)]

rng = rng_for("indomain-real")
reals = pd.concat([pick(kathbath[(kathbath.lang == code) & (kathbath.split == "test_indomain")], INDOMAIN_PER_LANGUAGE)
                   for code in LANGUAGES])
real_sample = subsample(reals, M4_EVAL_PER_LABEL, "lang", SEED)


def real_clip(r):
    return {"clip_id": f"kathbath_{r.lang}_{Path(r.fname).stem}", "dataset": "kathbath", "language": r.lang,
            "label": "real", "generator": "none", "speaker_id": r.speaker_id, "ref_speaker_id": r.speaker_id,
            "gender": r.gender[0].lower(), "text": r.text, "split": "test_indomain"}


n, downloaded = prepare(reals, real_sample, real_clip, lambda r: f"{Path(r.fname).stem}.flac", lambda r: 16000,
                        lambda todo: read_audio(todo, "audio_filepath"))
print(f"Kathbath test speakers: {len(reals):,} in the full set, {n:,} prepared ({downloaded:,} downloaded now), "
      f"{(time.time() - t0) / 60:.0f} min")

Kathbath test speakers: 3,000 in the full set, 500 prepared (0 downloaded now), 0 min


## 4. `test_unseen_gen`: Generators Never Seen in Training

All of our generated set (Milestone 2): Indic Parler-TTS, IndicF5 and MMS-TTS, every sentence spoken by all three.
The real side is Kathbath's recording of the same sentence by a Kathbath **test** speaker (not in IndicSynth or in
training), from Kathbath's own archive. So real and fake clips say the same words.

In [7]:
t0 = time.time()
lines = []
for path in fs.glob(f"{GENERATED}/manifests/*/*/*.jsonl"):
    lines += [json.loads(l) for l in fs.open(path).read().decode().splitlines() if l.strip()]
generated = pd.DataFrame(lines).drop_duplicates("clip_id")
generated = generated[~generated.flag_silent]
GENERATED_SR = {"indic-parler-tts": 44100, "indicf5": 24000, "mms-tts": 16000}

fake_sample = subsample(generated, M4_EVAL_PER_LABEL, ["lang", "generator"], SEED)


def generated_clip(r):
    return {"clip_id": r.clip_id, "dataset": "generated", "language": r.lang, "label": "fake",
            "generator": r.generator, "speaker_id": r.voice_id, "ref_speaker_id": r.voice_id,
            "gender": r.voice_gender, "text": r.text, "split": "test_unseen_gen"}


def fetch_generated(todo):
    with ThreadPoolExecutor(8) as pool:
        yield from zip(todo.itertuples(), pool.map(lambda p: fs.open(f"{GENERATED}/{p}").read(), todo.path))


n, downloaded = prepare(generated, fake_sample, generated_clip, lambda r: Path(r.path).name,
                        lambda r: GENERATED_SR[r.generator], fetch_generated)
print(f"Generated set: {len(generated):,} fakes in the full set, {n:,} prepared ({downloaded:,} downloaded now)")

sentences = generated.drop_duplicates(["lang", "kb_file"])[["lang", "kb_file", "text"]]
real_sample = subsample(sentences, M4_EVAL_PER_LABEL, "lang", SEED)


def test_speaker_clip(r):
    stem = Path(r.kb_file).stem
    speaker, gender = stem.split("-")[1:]
    return {"clip_id": f"kathbath_{r.lang}_{stem}", "dataset": "kathbath", "language": r.lang, "label": "real",
            "generator": "none", "speaker_id": speaker, "ref_speaker_id": speaker, "gender": gender, "text": r.text,
            "split": "test_unseen_gen"}


tar_path = REPO / "data" / "raw" / "kathbath" / "testunk_audio.tar"
if not tar_path.exists():
    tar_path.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["curl", "-s", "-o", str(tar_path), KATHBATH_TEST_TAR], check=True)


def fetch_test_speakers(todo):
    wanted = {f"kb_data_clean_m4a/{LANGUAGES[r.lang].lower()}/test/audio/{r.kb_file}": r for r in todo.itertuples()}
    with tarfile.open(tar_path) as tar:
        for member in tar:
            if member.name in wanted:
                yield wanted[member.name], tar.extractfile(member).read()


n, downloaded = prepare(sentences, real_sample, test_speaker_clip, lambda r: r.kb_file, lambda r: 44100,
                        fetch_test_speakers)
print(f"Kathbath test speakers: {len(sentences):,} in the full set, {n:,} prepared ({downloaded:,} decoded now), "
      f"{(time.time() - t0) / 60:.0f} min")

Generated set: 6,479 fakes in the full set, 500 prepared (0 downloaded now)


Kathbath test speakers: 2,160 in the full set, 500 prepared (0 decoded now), 1 min


## 5. `test_itw`: In-the-Wild (Real-World English)

In [8]:
t0 = time.time()
with zipfile.ZipFile(fs.open(f"{IN_THE_WILD}/release_in_the_wild.zip")) as zf:
    itw = pd.read_csv(zf.open("release_in_the_wild/meta.csv"))
itw["label"] = itw.label.map({"bona-fide": "real", "spoof": "fake"})
itw_sample = pd.concat([itw[itw.label == label].sample(M4_EVAL_PER_LABEL, random_state=SEED)
                        for label in ["real", "fake"]])


def itw_clip(r):
    return {"clip_id": f"itw_{Path(r.file).stem}", "dataset": "in_the_wild", "language": "en", "label": r.label,
            "generator": "unknown_tts" if r.label == "fake" else "none", "speaker_id": r.speaker,
            "ref_speaker_id": r.speaker, "gender": "", "text": "", "split": "test_itw"}


def fetch_itw(todo):
    if FULL:  # every file: download the whole archive once
        local = hf_hub_download("mueller91/In-The-Wild", "release_in_the_wild.zip", repo_type="dataset",
                                revision=IN_THE_WILD.split("@")[1])
        with zipfile.ZipFile(local) as zf:
            for r in todo.itertuples():
                yield r, zf.read(f"release_in_the_wild/{r.file}")
        return

    def read_one(name):  # a few files: one ranged read each
        with zipfile.ZipFile(fs.open(f"{IN_THE_WILD}/release_in_the_wild.zip")) as zf:
            return zf.read(f"release_in_the_wild/{name}")

    with ThreadPoolExecutor(8) as pool:
        yield from zip(todo.itertuples(), pool.map(read_one, todo.file))


n, downloaded = prepare(itw, itw_sample, itw_clip, lambda r: r.file, lambda r: 16000, fetch_itw)
print(f"In-the-Wild: {len(itw):,} in the full set, {n:,} prepared ({downloaded:,} downloaded now), "
      f"{(time.time() - t0) / 60:.0f} min")

In-the-Wild: 31,779 in the full set, 1,000 prepared (50 downloaded now), 1 min


## 6. `test_asvspoof`: ASVspoof 2019 LA Eval

Labels and attacks come from the official eval protocol.

In [9]:
t0 = time.time()
protocol = pd.read_csv(hf_hub_download("SpeechAntiSpoofingBenchmarks/ASVspoof2019_LA",
                                       "protocols/ASVspoof2019.LA.cm.eval.trl.txt", repo_type="dataset",
                                       revision=ASVSPOOF.split("@")[1]),
                       sep=" ", names=["speaker", "utt", "unused", "attack", "key"])
listing = CACHE / "asvspoof_eval_listing.parquet"
if listing.exists():
    asv = pd.read_parquet(listing)
else:
    with ThreadPoolExecutor(4) as pool:
        asv = pd.concat(pool.map(lambda p: read_columns(p, ["path"]), sorted(fs.glob(f"{ASVSPOOF}/data/test-*.parquet"))),
                        ignore_index=True)
    asv.to_parquet(listing)
asv["utt"] = asv.path.map(lambda p: Path(p).stem)
asv = asv.merge(protocol, on="utt")
asv["label"] = np.where(asv.key == "bonafide", "real", "fake")
asv_sample = pd.concat([asv[asv.label == "real"].sample(M4_EVAL_PER_LABEL, random_state=SEED),
                        subsample(asv[asv.label == "fake"], M4_EVAL_PER_LABEL, "attack", SEED)])


def asv_clip(r):
    return {"clip_id": f"asvspoof_{r.utt}", "dataset": "asvspoof2019_la", "language": "en", "label": r.label,
            "generator": r.attack if r.label == "fake" else "none", "speaker_id": r.speaker,
            "ref_speaker_id": r.speaker, "gender": "", "text": "", "split": "test_asvspoof"}


n, downloaded = prepare(asv, asv_sample, asv_clip, lambda r: f"{r.utt}.flac", lambda r: 16000,
                        lambda todo: read_audio(todo, "audio"))
print(f"ASVspoof eval: {len(asv):,} in the full set, {n:,} prepared ({downloaded:,} downloaded now), "
      f"{(time.time() - t0) / 60:.0f} min")

ASVspoof eval: 71,237 in the full set, 1,000 prepared (39 downloaded now), 0 min


## 7. `test_phone`: The In-Domain Test Through a Phone Channel

Each `test_indomain` clip is band-limited to 300–3,400 Hz, resampled through 8 kHz and coded with G.711 (mu-law or
A-law, chosen per clip with a fixed seed), using `src/augment.py`'s `phone_channel`. The result is brought back to
16 kHz and −20 dBFS like every other clip. Speakers, labels and generators stay those of the original clip.

In [10]:
t0 = time.time()
indomain = [r for r in rows if r["split"] == "test_indomain"]
made = 0
for r in indomain:
    clip_id = f"phone_{r['clip_id']}"
    out = PROCESSED / "phone" / r["language"] / f"{clip_id}.wav"
    if not out.exists():
        wave, _ = sf.read(DATA / r["path"], dtype="float32")
        codec_name = ("mulaw", "alaw")[zlib.crc32(clip_id.encode()) % 2]
        wave = normalise(phone_channel(wave, TARGET_SR, codec_name, FFMPEG))
        out.parent.mkdir(parents=True, exist_ok=True)
        sf.write(out, np.round(wave * 32767).astype(np.int16), TARGET_SR, subtype="PCM_16")
        made += 1
    pcm, _ = sf.read(out, dtype="int16")
    rows.append({**r, "clip_id": clip_id, "path": str(out.relative_to(DATA)), "raw_path": "", "split": "test_phone",
                 "duration_s": round(len(pcm) / TARGET_SR, 3), "orig_sr": TARGET_SR, "sha1": sha1(pcm)})
    if r["clip_id"] in m4_eval_ids:
        m4_eval_ids.add(clip_id)
print(f"Phone channel: {len(indomain):,} clips ({made:,} made now), {(time.time() - t0) / 60:.0f} min")

Phone channel: 1,000 clips (0 made now), 0 min


## 8. The Manifest

The test rows are added to the training manifest from notebook 1 (any older test rows are replaced). `m4_eval` marks
the Milestone-4 sample.

In [11]:
COLUMNS = ["clip_id", "path", "raw_path", "dataset", "language", "label", "generator", "speaker_id",
           "ref_speaker_id", "gender", "text", "duration_s", "orig_sr", "sha1", "split", "tune_subset", "train_eval",
           "m4_eval"]
TEST_SPLITS = ["test_indomain", "test_unseen_gen", "test_phone", "test_itw", "test_asvspoof"]
base = pd.read_csv(MANIFESTS / "manifest_m4.csv", keep_default_na=False, dtype={"speaker_id": str, "ref_speaker_id": str})
base = base[base.split.isin(["train", "valid"])]
tests = pd.DataFrame(rows).assign(tune_subset=False, train_eval=False)
tests["m4_eval"] = tests.clip_id.isin(m4_eval_ids)
manifest = pd.concat([base, tests[COLUMNS]], ignore_index=True)
manifest["split"] = pd.Categorical(manifest.split, ["train", "valid", *TEST_SPLITS])
manifest = manifest.sort_values(["split", "dataset", "language", "clip_id"])
manifest.to_csv(MANIFESTS / "manifest_m4.csv", index=False)

old_dropped = pd.read_csv(MANIFESTS / "dropped.csv", keep_default_na=False)
pd.concat([old_dropped[old_dropped.split.isin(["train", "valid"])], pd.DataFrame(dropped)]).to_csv(
    MANIFESTS / "dropped.csv", index=False)
print(f"{len(manifest):,} clips in the manifest ({len(tests):,} test clips, {tests.m4_eval.sum():,} in the "
      f"Milestone-4 sample); {len(dropped)} test clips dropped")

37,892 clips in the manifest (4,911 test clips, 4,911 in the Milestone-4 sample); 89 test clips dropped


### Test clips per split and label (all prepared, and the Milestone-4 sample)

In [12]:
test_rows = manifest[manifest.split.isin(TEST_SPLITS)]
pd.concat({"prepared": test_rows.pivot_table(index="split", columns="label", values="clip_id", aggfunc="count", observed=True),
           "m4_eval": test_rows[test_rows.m4_eval].pivot_table(index="split", columns="label", values="clip_id",
                                                               aggfunc="count", observed=True)}, axis=1)

prepared      m4_eval     
label               fake real    fake real
split                                     
test_indomain        500  500     500  500
test_unseen_gen      500  500     500  500
test_phone           500  500     500  500
test_itw             478  472     478  472
test_asvspoof        478  483     478  483

### The Milestone-4 sample per language and generator

In [13]:
sample = test_rows[test_rows.m4_eval]
sample.pivot_table(index=["split", "language"], columns="label", values="clip_id", aggfunc="count", fill_value=0,
                   observed=True)

label                     fake  real
split           language            
test_indomain   hi         168   166
                ta         166   167
                te         166   167
test_unseen_gen hi         165   166
                ta         168   167
                te         167   167
test_phone      hi         168   166
                ta         166   167
                te         166   167
test_itw        en         478   472
test_asvspoof   en         478   483

In [14]:
sample.groupby(["split", "generator"], observed=True).size().unstack(0).fillna(0).astype(int)

split,test_indomain,test_unseen_gen,test_phone,test_itw,test_asvspoof
generator,,,,,
A07,0,0,0,0,41
A08,0,0,0,0,35
A09,0,0,0,0,37
A10,0,0,0,0,36
A11,0,0,0,0,36
A12,0,0,0,0,37
A13,0,0,0,0,36
A14,0,0,0,0,38
A15,0,0,0,0,37


## 9. Leakage Checks

No test speaker may be a training or validation speaker (including FreeVC24 source speakers), no clip may appear
twice, and ASVspoof eval speakers must not appear in its train or dev partitions. Sentence overlap between the
unseen-generator test and `train` is reported, not required to be 0 (Kathbath has several people read the same
sentence).

In [15]:
def speaker_set(splits):
    part = manifest[manifest.split.isin(splits) & manifest.language.isin(LANGUAGES)]
    return set(part.speaker_id.astype(str)) | set(part.ref_speaker_id.astype(str))


seen = speaker_set(["train", "valid"])
asv_rows = manifest[manifest.dataset == "asvspoof2019_la"]
train_text = set(manifest[(manifest.split == "train") & (manifest.text != "")].text)
leakage = pd.DataFrame([
    ("test_indomain speakers also in train or valid", len(speaker_set(["test_indomain"]) & seen)),
    ("test_unseen_gen real speakers also in train or valid",
     len(set(manifest[(manifest.split == "test_unseen_gen") & (manifest.dataset == "kathbath")].speaker_id) & seen)),
    ("duplicate clips (same SHA-1)", int(manifest.sha1.duplicated().sum())),
    ("ASVspoof eval speakers also in train or dev",
     len(set(asv_rows[asv_rows.split == "test_asvspoof"].speaker_id) & set(asv_rows[asv_rows.split.isin(["train", "valid"])].speaker_id))),
    ("test_unseen_gen sentences also in train (reported only)",
     len(set(manifest[manifest.split == "test_unseen_gen"].text) & train_text)),
], columns=["check", "count"])
leakage.to_csv(MANIFESTS / "leakage_report_test.csv", index=False)
leakage

,check,count
0,test_indomain speakers also in train or valid,0
1,test_unseen_gen real speakers also in train or...,0
2,duplicate clips (same SHA-1),0
3,ASVspoof eval speakers also in train or dev,0
4,test_unseen_gen sentences also in train (repor...,0


## 10. Folder Structure

In [16]:
for folder in sorted(p for p in DATA.glob("*/*") if p.is_dir()):
    files = [f for f in folder.rglob("*") if f.is_file()]
    print(f"{str(folder.relative_to(DATA)):35s} {len(files):6,} files {sum(f.stat().st_size for f in files) / 1e9:6.2f} GB")

processed/asvspoof2019_la            6,946 files   0.56 GB
processed/generated                    500 files   0.09 GB
processed/in_the_wild                  950 files   0.14 GB
processed/indicsynth                13,996 files   3.05 GB


processed/kathbath                  14,500 files   2.85 GB
processed/phone                      1,000 files   0.21 GB
raw/asvspoof2019_la                     20 files   0.00 GB
raw/generated                           30 files   0.01 GB
raw/in_the_wild                         10 files   0.00 GB
raw/indicsynth                          60 files   0.02 GB
raw/kathbath                            60 files   0.01 GB


## 11. Summary

The test sets are added to `data/m4/` and its manifest, and the Kaggle dataset `dsai-group3-m4-data` is updated to a
new version. Milestone 4 scores the `m4_eval` sample of each test split; running this notebook again with `M4_FULL=1`
prepares the remaining clips of the full sets for Milestone 5.